In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import optax
from tqdm import tqdm
from functools import partial

# ==============================================================================
# 1. 全局参数 & H₂ 分子定义
# ==============================================================================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

# ==============================================================================
# 2. 神经网络 Ansatz (Flax Linen 版本)
# ==============================================================================
class SingleStateAnsatz(nn.Module):
    hidden_dim: int = 16
    @nn.compact
    def __call__(self, x):
        x = x.astype(jnp.complex128)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(self.hidden_dim, param_dtype=jnp.complex128)(x)
        x = nn.tanh(x)
        x = nn.Dense(1, param_dtype=jnp.complex128)(x)
        return x.squeeze(-1)

# ==============================================================================
# 3. 辅助函数（使用全局 ha，并保留 JIT）
# ==============================================================================
@partial(jax.jit, static_argnames=('model',))
def compute_local_energies(params, model, sigma):
    eta, H_sigmaeta = ha.get_conn_padded(sigma)
    logpsi_sigma = model.apply(params, sigma)
    logpsi_eta = model.apply(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_sigmaeta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)

@partial(jax.jit, static_argnames=('model',))
def estimate_energy(params, model, sigma):
    E_loc = compute_local_energies(params, model, sigma)
    return nk.stats.Stats(
        mean=jnp.mean(E_loc),
        error_of_mean=jnp.sqrt(jnp.var(E_loc) / E_loc.size),
        variance=jnp.var(E_loc)
    )

@partial(jax.jit, static_argnames=('model',))
def energy_and_grad(params, model, samples):
    def loss_fn(p):
        E_loc = compute_local_energies(p, model, samples)
        E = jnp.mean(E_loc)
        return E.real, E
    (_, E), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
    return E, grads

# ==============================================================================
# 4. 初始化模型、采样器、优化器
# ==============================================================================
model = SingleStateAnsatz(hidden_dim=12)

dummy_input = hi.random_state(jax.random.key(0), size=1)
params = model.init(jax.random.key(21), dummy_input)

# 采样器
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)
sampler_state = sampler.init_state(model, params, seed=1)

optimizer = optax.adam(learning_rate=1e-2)
opt_state = optimizer.init(params)



/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


In [ ]:
class NESMCState(nk.vqs.MCState):
    """自定义 MCState，通过重写 local_estimators 实现 NES-VMC 逻辑"""
    
    def local_estimators(self, op, *, chunk_size=None):
        """
        重写父类方法，返回自定义的局域能量（迹）。
        注意：op 参数在此被忽略，因为 NES-VMC 的局域能量是迹，而非 H_loc。
        """
        print('local_estimators')
        # 获取当前样本。这里使用 self.samples 而不是重新采样。
        samples = self.samples
        
        # 获取当前参数并重建模型
        # 注意：self.parameters 的格式是 (graphdef, variables)
        graphdef, variables = self.parameters
        model = nnx.merge(graphdef, variables)
        
        # 调用自定义函数计算每个样本的迹
        local_energies = batch_local_energy(ha, model, samples)
        
        # 返回形状为 (n_chains, n_samples_per_chain) 的数组
        n_chains = self.sampler.n_chains
        chain_length = samples.shape[0] // n_chains
        return local_energies.reshape(n_chains, chain_length)

In [3]:
class NESMCState(nk.vqs.MCState):
    def expect_and_grad(self, O, *, mutable=None, use_covariance=None):
        samples = self.samples
        model = self.model  # nnx 模型已包含最新参数
        local_energies = batch_local_energy(ha, model, samples)
        loss = local_energies.mean()
        # 手动计算梯度
        def loss_fn(vars): ...
        grad_vars = jax.grad(loss_fn)(nnx.state(model))
        return nk.stats.Stats(mean=loss), (None, grad_vars)

In [4]:
# 使用自定义的 NESMCState 替代 MCState
vstate = NESMCState(
    sampler,
    model,
    n_samples=1024,
    n_discard_per_chain=16,
    seed=123
)
optimizer = nk.optimizer.Sgd(learning_rate=0.1)

gs = nk.driver.VMC(
    ha,
    optimizer,
    variational_state=vstate
)

In [5]:
gs.run(n_iter=10)

  0%|          | 0/10 [00:00<?, ?it/s]


NameError: name 'batch_local_energy' is not defined

In [ ]:
# ==============================================================================
# 8. 提取激发态能量
# ==============================================================================
K = 2
vstate.n_samples = 4096
vstate.n_discard_per_chain = 32
samples_final = vstate.sample()
samples_flat = samples_final.reshape(-1, K * 4)

graphdef, variables = vstate.parameters
model_final = nnx.merge(graphdef, variables)

def compute_local_energy_matrix_full(ham, model, x_single):
    x_batch = x_single.reshape(model.K, model.n_spin)
    M = jnp.array([
        [model.single_ansatz_list[j](x_batch[i]) for j in range(model.K)]
        for i in range(model.K)
    ])
    Hp = Ham_Psi(ham, model, x_batch)
    M_reg = M + 1e-8 * jnp.eye(model.K, dtype=M.dtype)
    return jnp.linalg.solve(M_reg, Hp)

batch_emat = jax.vmap(compute_local_energy_matrix_full, in_axes=(None, None, 0))
E_mats = batch_emat(ha, model_final, samples_flat)
E_mat_avg = jnp.mean(E_mats, axis=0)

eigvals = jnp.linalg.eigvalsh(E_mat_avg).real
print("\n" + "=" * 60)
print("NES-VMC 计算结果（方案1：继承 MCState + 重写 local_estimators）")
for i, e in enumerate(eigvals):
    print(f"E{i} = {e:.8f} Ha  |  与 FCI 偏差: {e - E_fcis[i]:.6f} Ha")

In [ ]:
# 加这行诊断
print("Type of vstate.parameters:", type(vstate.parameters))
print("Content (truncated):", repr(vstate.parameters)[:200])

# 注释掉原来报错的行
# graphdef, variables = vstate.parameters
# model_final = nnx.merge(graphdef, variables)｜

In [ ]:
vstate.expect()

In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import optax
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx

# 临时禁用 JIT 以观察 print
jax.config.update('jax_disable_jit', False)  # 保持 True 可看 print

# ==============================================================================
# 1. H₂ 分子与 FCI 基准
# ==============================================================================
K = 2
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, _ = cisolver.kernel()
print("=" * 60)
print("FCI 基准能量 (Ha)")
for i, e in enumerate(E_fcis):
    print(f"E{i} = {e:.8f}  |  激发能 = {(e - E_fcis[0]) * 27.2114:.4f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)

# ==============================================================================
# 2. 扩展希尔伯特空间与采样器
# ==============================================================================
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)
hi_ext = hi ** K

edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)
sampler = nk.sampler.MetropolisSampler(
    hi_ext,
    rule=tensor_rule,
    n_chains=16,
    sweep_size=32,
    dtype=jnp.int8
)

# ==============================================================================
# 3. 模型定义
# ==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

class NESModel(nnx.Module):
    def __init__(self, n_spin_orbitals: int, K: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.K = K
        self.n_spin = n_spin_orbitals
        keys = jax.random.split(rngs.key, K)
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(keys[i]))
            for i in range(K)
        ]

    def __call__(self, x_batch):
        batch_size = x_batch.shape[0]
        x_reshaped = x_batch.reshape(batch_size, self.K, self.n_spin)

        def log_psi_per_sample(x_sample):
            M = jnp.array([
                [self.single_ansatz_list[j](x_sample[i]) for j in range(self.K)]
                for i in range(self.K)
            ])
            det = jnp.linalg.det(M)
            return jnp.log(det + 1e-10)

        log_psi_vec = jax.vmap(log_psi_per_sample)(x_reshaped)
        return log_psi_vec

# ==============================================================================
# 4. 局域能量计算（迹）
# ==============================================================================
def Ham_psi(ham, single_ansatz, x):
    x_primes, mels = ham.get_conn_padded(x)
    psi_vals = jax.vmap(single_ansatz)(x_primes)
    return jnp.sum(mels * psi_vals)

def Ham_Psi(ham, model, x_batch):
    K_state = model.K
    H_mat = []
    for i in range(K_state):
        row = []
        for j in range(K_state):
            row.append(Ham_psi(ham, model.single_ansatz_list[j], x_batch[i]))
        H_mat.append(row)
    return jnp.array(H_mat, dtype=complex)

def compute_local_energy_single(ham, model, x_single):
    x_batch = x_single.reshape(model.K, model.n_spin)
    M = jnp.array([
        [model.single_ansatz_list[j](x_batch[i]) for j in range(model.K)]
        for i in range(model.K)
    ])
    Hp = Ham_Psi(ham, model, x_batch)
    M_reg = M + 1e-8 * jnp.eye(model.K, dtype=M.dtype)
    E_mat = jnp.linalg.solve(M_reg, Hp)
    return jnp.trace(E_mat).real

batch_local_energy = jax.vmap(compute_local_energy_single, in_axes=(None, None, 0))

# ==============================================================================
# 5. 自定义 MCState，重写 expect_and_grad
# ==============================================================================
class NESMCState(nk.vqs.MCState):
    def expect_and_grad(self, O, *, mutable=None, use_covariance=None):
        print(">>> NESMCState.expect_and_grad called")
        
        samples = self.samples
        graphdef, variables = self.parameters
        model = nnx.merge(graphdef, variables)
        
        local_energies = batch_local_energy(ha, model, samples)
        loss = local_energies.mean()
        print(f"   Loss (trace avg): {loss:.6f}")
        
        def loss_fn(vars):
            merged_model = nnx.merge(graphdef, vars)
            return batch_local_energy(ha, merged_model, samples).mean()
        
        grad_vars = jax.grad(loss_fn)(variables)
        grads = (None, grad_vars)
        loss_stats = nk.stats.Stats(mean=loss, variance=0.0, error_of_mean=0.0)
        return loss_stats, grads

# ==============================================================================
# 6. 训练
# ==============================================================================
model = NESModel(n_spin_orbitals=4, K=K, hidden_dim=16, rngs=nnx.Rngs(42))

vstate = NESMCState(
    sampler,
    model,
    n_samples=1024,
    n_discard_per_chain=16,
    seed=123
)

optimizer = nk.optimizer.Sgd(learning_rate=0.1)

gs = nk.driver.VMC(
    ha,
    optimizer,
    variational_state=vstate
)

print("\n开始训练...")
gs.run(n_iter=10)